# Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Dataset, Subset
import random
import numpy as np

# Hyperparameters

In [ ]:
ROOT = './data'     # path to store data
BATCH_SIZE = 256    # batch size
EPOCHS = 100        # total epochs
PROJ_DIM = 128      # projection head dimension, this corresponds to the embedding size
LR = 1e-3           # learning rate
TEMPERATURE = 0.17  # temperature for contrastive loss
LINEAR_EPOCHS = 20  # epochs to train linear classifier (used for evaluation)
EVAL_INTERVAL = 1   # evaluate every N epochs (including epoch 
SUBSET_RATIO = 0.1  # fraction of train set for linear eval
FREEZE_EPOCHS = 20  # epochs to train only projection head, i.e. freeze ResNet

# Functional Invariances

In [ ]:
# Data transforms, these correspond to the functional invariances we assume
def get_contrastive_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(32),
        # transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
        # transforms.RandomGrayscale(p=0.2),
        # transforms.GaussianBlur(kernel_size=3),
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
    ])

# Evaluation transforms (no resize)
EVAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

# Get Data

In [ ]:
class ContrastiveCIFAR10(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.dataset = datasets.CIFAR10(root=root, train=train, download=True)
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        return self.transform(img), self.transform(img)

# Model

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim, proj_dim)
        )
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    def __init__(self, proj_dim):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1
        backbone = resnet18(weights=weights)
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.proj = ProjectionHead(feat_dim, proj_dim)
    def forward(self, x):
        h = self.backbone(x)
        z = self.proj(h)
        return F.normalize(z, dim=1)

# Unsupervised Loss

In [ ]:
# NT-Xent loss
def nt_xent_loss(z_i, z_j, temperature=TEMPERATURE):
    B = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)
    sim = torch.matmul(z, z.T) / temperature
    mask = ~torch.eye(2*B, device=z.device).bool()
    exp_sim = torch.exp(sim) * mask
    positives = torch.exp((z_i * z_j).sum(dim=-1) / temperature)
    positives = torch.cat([positives, positives], dim=0)
    loss = -torch.log(positives / exp_sim.sum(dim=-1))
    return loss.mean()



# Utils
Functions for extracting embeddings and training linear classifier

In [ ]:
# Embedding extraction
def extract_embeddings(model, dataset, device, batch_size):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    embs, labs = [], []
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            h = model.backbone(imgs)
            z = model.proj(h)
            embs.append(z.cpu())
            labs.append(targets)
    return torch.cat(embs), torch.cat(labs)

# Linear evaluation with classwise accuracy
def train_and_eval_linear(train_embs, train_labels, test_embs, test_labels, device):
    clf = nn.Linear(train_embs.size(1), 10).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    train_embs, train_labels = train_embs.to(device), train_labels.to(device)
    test_embs, test_labels = test_embs.to(device), test_labels.to(device)
    for _ in range(LINEAR_EPOCHS):
        clf.train()
        logits = clf(train_embs)
        loss = crit(logits, train_labels)
        opt.zero_grad(); loss.backward(); opt.step()
    clf.eval()
    with torch.no_grad():
        logits = clf(test_embs)
        preds = logits.argmax(dim=1)
        correct = preds == test_labels
        overall_acc = correct.float().mean().item()
        class_acc = {cls: correct[test_labels==cls].float().mean().item()
                     for cls in range(10)}
    return overall_acc, class_acc

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Datasets
contrast_ds = ContrastiveCIFAR10(ROOT, train=True, transform=get_contrastive_transforms())
contrast_loader = DataLoader(contrast_ds, batch_size=BATCH_SIZE,
                                shuffle=True, num_workers=2)
train_ds = datasets.CIFAR10(ROOT, train=True, download=True, transform=EVAL_TRANSFORM)
test_ds  = datasets.CIFAR10(ROOT, train=False,download=True, transform=EVAL_TRANSFORM)

# Model
model = SimCLR(PROJ_DIM).to(device)

# Optimizers: separate for backbone and projection head
backbone_opt = torch.optim.Adam(
    model.backbone.parameters(), lr=LR*0.1, weight_decay=1e-4)
proj_opt     = torch.optim.Adam(
    model.proj.parameters(),     lr=LR,       weight_decay=1e-6)

# LR schedulers (cosine annealing)
backbone_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    backbone_opt, T_max=max(1, EPOCHS-FREEZE_EPOCHS))
proj_sched     = torch.optim.lr_scheduler.CosineAnnealingLR(
    proj_opt,     T_max=EPOCHS)

n = len(train_ds)
subset_size = int(SUBSET_RATIO * n)

for epoch in range(EPOCHS + 1):
    # Contrastive training
    if epoch > 0:
        model.train()
        total_loss = 0
        for x_i, x_j in contrast_loader:
            x_i, x_j = x_i.to(device), x_j.to(device)
            z_i, z_j = model(x_i), model(x_j)
            loss = nt_xent_loss(z_i, z_j)
            # zero grads
            backbone_opt.zero_grad()
            proj_opt.zero_grad()
            loss.backward()
            # update
            if epoch <= FREEZE_EPOCHS:
                proj_opt.step()
            else:
                backbone_opt.step()
                proj_opt.step()
            total_loss += loss.item()
        print(f"Epoch {epoch}/{EPOCHS} Contrastive Loss: {total_loss/len(contrast_loader):.4f}")

    # Linear eval
    if epoch % EVAL_INTERVAL == 0:
        indices = random.sample(range(n), subset_size)
        sub_ds = Subset(train_ds, indices)
        train_embs, train_lbls = extract_embeddings(model, sub_ds,    device, BATCH_SIZE)
        test_embs,  test_lbls  = extract_embeddings(model, test_ds,   device, BATCH_SIZE)
        overall_acc, class_acc = train_and_eval_linear(
            train_embs, train_lbls, test_embs, test_lbls, device)
        print(f"Linear eval @ epoch {epoch}: Overall = {overall_acc*100:.2f}%")
        for cls, acc in class_acc.items():
            print(f"    Class {cls}: {acc*100:.2f}%")
        # Save embeddings every 10 epochs
        if epoch % 10 == 0:
            filename = f"embeddings_epoch{epoch}_acc{overall_acc*100:.2f}.npz"
            np.savez_compressed(filename,
                                    embeddings=test_embs.numpy(),
                                    labels=test_lbls.numpy())
            print(f"Saved embeddings to {filename}")

    # step LR schedulers
    proj_sched.step()
    if epoch > FREEZE_EPOCHS:
        backbone_sched.step()

Linear eval @ epoch 0: Overall = 26.51%
    Class 0: 15.90%
    Class 1: 32.30%
    Class 2: 46.60%
    Class 3: 15.60%
    Class 4: 9.40%
    Class 5: 47.40%
    Class 6: 5.40%
    Class 7: 30.50%
    Class 8: 56.40%
    Class 9: 5.60%
Saved embeddings to embeddings_epoch0_acc26.51.npz
Epoch 1/100 Contrastive Loss: 5.3434
Linear eval @ epoch 1: Overall = 31.42%
    Class 0: 31.60%
    Class 1: 35.60%
    Class 2: 20.50%
    Class 3: 11.80%
    Class 4: 8.50%
    Class 5: 28.90%
    Class 6: 63.20%
    Class 7: 38.60%
    Class 8: 43.60%
    Class 9: 31.90%
Epoch 2/100 Contrastive Loss: 5.1781
Linear eval @ epoch 2: Overall = 30.49%
    Class 0: 18.00%
    Class 1: 49.50%
    Class 2: 19.20%
    Class 3: 20.20%
    Class 4: 25.80%
    Class 5: 17.20%
    Class 6: 56.00%
    Class 7: 41.70%
    Class 8: 41.40%
    Class 9: 15.90%
Epoch 3/100 Contrastive Loss: 5.1210
Linear eval @ epoch 3: Overall = 32.69%
    Class 0: 21.80%
    Class 1: 28.70%
    Class 2: 19.40%
    Class 3: 12.20%
  